# Lecture 06 - Logistic Regression

Mục tiêu:
1. Xây dựng pipeline tiền xử lý và so sánh các mô hình Logistic Regression có và không sử dụng class-balanced weights.
2. Giải thích mô hình Logistic Regression được huấn luyện với L1 regularisation và thảo luận ý nghĩa thực tiễn của các hệ số.
3. Thiết kế chiến lược sàng lọc rủi ro cho ngân hàng sử dụng dự đoán xác suất vỡ nợ và phân tích sự đánh đổi giữa precision-recall.

Cách làm bài:

Bạn làm bài ở những chỗ ghi là:
```python
# YOUR CODE HERE
raise NotImplementedError()
```
hoặc đối với markdown cell thì là:
```markdown
YOUR ANSWER HERE
```
Khi làm bài thì xóa dòng `raise NotImplementedError()`. Ở dưới các cell được yêu cầu làm bài thường sẽ có cell TEST, bạn cần vượt qua được các cell này.

## Import các thư viện cần thiết

In [ ]:
import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score

## Câu hỏi 1: Tiền xử lý dữ liệu và xây dựng mô hình giải quyết mất cân bằng dữ liệu

### Đọc dữ liệu

Tập dữ liệu này chứa thông tin về tình trạng vỡ nợ, các yếu tố nhân khẩu học, dữ liệu tín dụng, lịch sử thanh toán và sao kê hóa đơn của khách hàng sử dụng thẻ tín dụng tại Đài Loan từ tháng 4 đến tháng 9 năm 2005.

* ID: ID của từng khách hàng.
* LIMIT_BAL: Hạn mức tín dụng được cấp (tính bằng Đài tệ (NT dollars), bao gồm tín dụng cá nhân và tín dụng gia đình/người phụ thuộc).
* SEX: Giới tính (1=nam, 2=nữ).
* EDUCATION: Trình độ học vấn (1 = cao học, 2 = đại học, 3 = trung học phổ thông, 0/4/5/6 = khác).
* MARRIAGE: Tình trạng hôn nhân (1 = đã kết hôn, 2 = độc thân, 3 = đã ly hôn, 0 = khác).
* AGE: Tuổi (tính theo năm).
* Lịch sử thanh toán (PAY_X): Trạng thái trả nợ trong các tháng tương ứng (-2=Không tiêu dùng(No consumption), -1=Đã thanh toán đầy đủ (Balance paid in full), 0=Sử dụng tín dụng quay vòng (The use of revolving credit), 1=Trễ hạn 1 tháng,... 8=Trễ hạn 8 tháng, 9=Trễ hạn từ 9 tháng trở lên):
    * PAY_0: Tháng 9/2005.
    * PAY_2: Tháng 8/2005.
    * PAY_3: Tháng 7/2005.
    * PAY_4: Tháng 6/2005.
    * PAY_5: Tháng 5/2005.
    * PAY_6: Tháng 4/2005.
* Sao kê hóa đơn (BILL_AMTX): (Tính bằng Đài tệ (NT dollars)):
    * BILL_AMT1: Tháng 9/2005.
    * BILL_AMT2: Tháng 8/2005.
    * BILL_AMT3: Tháng 7/2005.
    * BILL_AMT4: Tháng 6/2005.
    * BILL_AMT5: Tháng 5/2005.
    * BILL_AMT6: Tháng 4/2005.
* Số tiền đã thanh toán trước đó (PAY_AMTX): (Tính bằng Đài tệ (NT dollars))
    * PAY_AMT1: Tháng 9/2005.
    * PAY_AMT2: Tháng 8/2005.
    * PAY_AMT3: Tháng 7/2005.
    * PAY_AMT4: Tháng 6/2005.
    * PAY_AMT5: Tháng 5/2005.
    * PAY_AMT6: Tháng 4/2005.
* default.payment.next.month: Khách hàng có vỡ nợ trong tháng kế tiếp hay không (Default payment (1=yes, 0=no)).

In [2]:
data = pd.read_csv('data/UCI_Credit_Card.csv')
data.head(5)

,ID,LIMIT_BAL,SEX,EDUCATION,MARRIAGE,AGE,PAY_0,PAY_2,PAY_3,PAY_4,...,BILL_AMT4,BILL_AMT5,BILL_AMT6,PAY_AMT1,PAY_AMT2,PAY_AMT3,PAY_AMT4,PAY_AMT5,PAY_AMT6,default.payment.next.month
0,1,20000.0,2,2,1,24,2,2,-1,-1,...,0.0,0.0,0.0,0.0,689.0,0.0,0.0,0.0,0.0,1
1,2,120000.0,2,2,2,26,-1,2,0,0,...,3272.0,3455.0,3261.0,0.0,1000.0,1000.0,1000.0,0.0,2000.0,1
2,3,90000.0,2,2,2,34,0,0,0,0,...,14331.0,14948.0,15549.0,1518.0,1500.0,1000.0,1000.0,1000.0,5000.0,0
3,4,50000.0,2,2,1,37,0,0,0,0,...,28314.0,28959.0,29547.0,2000.0,2019.0,1200.0,1100.0,1069.0,1000.0,0
4,5,50000.0,1,2,1,57,-1,0,-1,0,...,20940.0,19146.0,19131.0,2000.0,36681.0,10000.0,9000.0,689.0,679.0,0


### Tiền xử lý và khám phá dữ liệu

In [3]:
data.describe()

,ID,LIMIT_BAL,SEX,EDUCATION,MARRIAGE,AGE,PAY_0,PAY_2,PAY_3,PAY_4,...,BILL_AMT4,BILL_AMT5,BILL_AMT6,PAY_AMT1,PAY_AMT2,PAY_AMT3,PAY_AMT4,PAY_AMT5,PAY_AMT6,default.payment.next.month
count,30000.000000,30000.000000,30000.000000,30000.000000,30000.000000,30000.000000,30000.000000,30000.000000,30000.000000,30000.000000,...,30000.000000,30000.000000,30000.000000,30000.000000,3.000000e+04,30000.00000,30000.000000,30000.000000,30000.000000,30000.000000
mean,15000.500000,167484.322667,1.603733,1.853133,1.551867,35.485500,-0.016700,-0.133767,-0.166200,-0.220667,...,43262.948967,40311.400967,38871.760400,5663.580500,5.921163e+03,5225.68150,4826.076867,4799.387633,5215.502567,0.221200
std,8660.398374,129747.661567,0.489129,0.790349,0.521970,9.217904,1.123802,1.197186,1.196868,1.169139,...,64332.856134,60797.155770,59554.107537,16563.280354,2.304087e+04,17606.96147,15666.159744,15278.305679,17777.465775,0.415062
min,1.000000,10000.000000,1.000000,0.000000,0.000000,21.000000,-2.000000,-2.000000,-2.000000,-2.000000,...,-170000.000000,-81334.000000,-339603.000000,0.000000,0.000000e+00,0.00000,0.000000,0.000000,0.000000,0.000000
25%,7500.750000,50000.000000,1.000000,1.000000,1.000000,28.000000,-1.000000,-1.000000,-1.000000,-1.000000,...,2326.750000,1763.000000,1256.000000,1000.000000,8.330000e+02,390.00000,296.000000,252.500000,117.750000,0.000000
50%,15000.500000,140000.000000,2.000000,2.000000,2.000000,34.000000,0.000000,0.000000,0.000000,0.000000,...,19052.000000,18104.500000,17071.000000,2100.000000,2.009000e+03,1800.00000,1500.000000,1500.000000,1500.000000,0.000000
75%,22500.250000,240000.000000,2.000000,2.000000,2.000000,41.000000,0.000000,0.000000,0.000000,0.000000,...,54506.000000,50190.500000,49198.250000,5006.000000,5.000000e+03,4505.00000,4013.250000,4031.500000,4000.000000,0.000000
max,30000.000000,1000000.000000,2.000000,6.000000,3.000000,79.000000,8.000000,8.000000,8.000000,8.000000,...,891586.000000,927171.000000,961664.000000,873552.000000,1.684259e+06,896040.00000,621000.000000,426529.000000,528666.000000,1.000000


In [4]:
data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 30000 entries, 0 to 29999
Data columns (total 25 columns):
 #   Column                      Non-Null Count  Dtype  
---  ------                      --------------  -----  
 0   ID                          30000 non-null  int64  
 1   LIMIT_BAL                   30000 non-null  float64
 2   SEX                         30000 non-null  int64  
 3   EDUCATION                   30000 non-null  int64  
 4   MARRIAGE                    30000 non-null  int64  
 5   AGE                         30000 non-null  int64  
 6   PAY_0                       30000 non-null  int64  
 7   PAY_2                       30000 non-null  int64  
 8   PAY_3                       30000 non-null  int64  
 9   PAY_4                       30000 non-null  int64  
 10  PAY_5                       30000 non-null  int64  
 11  PAY_6                       30000 non-null  int64  
 12  BILL_AMT1                   30000 non-null  float64
 13  BILL_AMT2                   300

#### Xử lý các giá trị không hợp lệ
Dựa trên ý nghĩa, thống kê mô tả (describe) và thông tin tổng quát (info) của các đặc trưng trong tập dữ liệu, ta thực hiện một số bước xử lý như sau:
- Đổi tên cột `PAY_0` thành `PAY_1` để thống nhất với các cột `PAY_2`, `PAY_3`, `BILL_AMT1`... và giúp dữ liệu dễ đọc, dễ phân tích hơn.
- Đổi tên cột `default.payment.next.month` thành `default` để ngắn gọn hơn.
- Chuẩn hóa giá trị trong cột `EDUCATION`: Các giá trị 0, 4, 5, 6 đều mang ý nghĩa "khác" (other), do đó ta gộp chúng lại thành một giá trị duy nhất là 0 để đơn giản hóa phân tích.
- Không có giá trị thiếu (null) trong tập dữ liệu nên không cần xử lý thiếu dữ liệu.
- Loại bỏ các dòng bị trùng lặp (nếu có) để đảm bảo dữ liệu huấn luyện không bị lệch do thông tin lặp lại.

In [ ]:
# YOUR CODE HERE
raise NotImplementedError()
data.rename(...)
data['EDUCATION'] = ...
data = ...

In [ ]:
# TEST
assert data.shape == (30000, 25)
assert 'PAY_1' in data.columns
assert np.array_equal(data['EDUCATION'].unique(), [2, 1, 3, 0])

#### Biến categorical thành one-hot encoding
Sử dụng [pd.get_dummies()](https://pandas.pydata.org/pandas-docs/stable/reference/api/pandas.get_dummies.html) để chuyển dữ liệu categorical thành one-hot encoding

In [ ]:
cat_cols = ['SEX', 'EDUCATION', 'MARRIAGE'] + [f'PAY_{i}' for i in range(1, 7)]
data = pd.get_dummies(data, columns=cat_cols, drop_first=True)

### Xây dựng mô hình

Chia tập train/test theo tỷ lệ 80/20

In [ ]:
X = data.drop(['ID', 'default'], axis=1)
y = data['default']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
# Tạo bản sao để cập nhật các cột đã chuẩn hóa.
X_train = X_train.copy()
X_test = X_test.copy()


#### Chuẩn hóa sau khi chia train/test
Chỉ fit `StandardScaler()` trên các biến liên tục của tập train, rồi dùng cùng scaler để transform tập test. Không fit trên toàn bộ dữ liệu để tránh rò rỉ thông tin từ test.


In [ ]:
# YOUR CODE HERE
raise NotImplementedError()
cont_cols = ...
scaler = ...
X_train[cont_cols] = ...
X_test[cont_cols] = ...

In [ ]:
# TEST
assert len(cont_cols) == 14
assert np.allclose(X_train[cont_cols].mean(), 0, atol=1e-10)
assert np.allclose(X_train[cont_cols].std(ddof=0), 1, atol=1e-10)
# Test phải được transform bằng scaler đã fit trên train.
assert np.allclose(X_test[cont_cols], scaler.transform(X.loc[X_test.index, cont_cols]))


#### Xây dựng mô hình Logistic Regression

In [ ]:
y_train.value_counts()

**Vấn đề mất cân bằng dữ liệu:**
Ta có thể thấy số lượng nhãn 0 và nhãn 1 trong tập dữ liệu huấn luyện chênh lệch nhau khá nhiều (nhãn 0 gấp nhãn 1 $\approx 3.5$ lần). Mô hình có thể dự đoán toàn bộ nhãn là 0 và vẫn đạt được accuracy$\approx 77.8%$ nhưng lại bỏ qua nhãn 1 là nhãn quan trọng. Nếu tỉ số số lượng giữa hai nhãn là 9:1, accuracy sẽ lên tới 90% trong khi không dự đoán được ai sẽ vỡ nợ (nhãn 1).

Một trong những giải pháp cho vấn đề này là gán trọng số cho các lớp: Những lớp hiếm được gán trọng số lớn hơn, giúp tăng độ ưu tiên khi mô hình học những lớp này.

Trong các mô hình phân loại như Logistic Regression mà scikit-learn cung cấp, tham số [class_weight](https://scikit-learn.org/stable/modules/generated/sklearn.linear_model.LogisticRegression.html) cho phép gán trọng số khác nhau cho từng lớp trong quá trình huấn luyện. Các giá trị `class-weight` có thể có:
- `None` (mặc định): Tất cả các lớp có trọng số 1, nghĩa là các lớp được xem là như nhau bất kể tần suất của chúng.
- `balanced`: Điều chỉnh trọng số tỷ lệ nghịch với tần suất xuất hiện. Trọng số của một lớp sẽ bằng $n\_samples / (n\_classes * np.bincount(y))$. Trong đó:
    - $n\_samples$: Số mẫu.
    - $n\_classes$: Số lớp.
    - $np.bincount(y)$: Số mẫu của từng lớp.
- dict: Tùy chỉnh trọng số cho từng lớp: keys là các nhãn và values là trọng số của các nhãn tương ứng. Ví dụ: class_weight = {0: 1.0, 1:3.5}.


**Nhiệm vụ của bạn:** Xây dựng mô hình logistic regression không sử dụng class_weight và mô hình logistic regression có sử dụng class_weight='balanced', rồi tiến hành so sánh Accuracy và F1-score giữa hai mô hình.

In [ ]:
# Mô hình không sử dụng class_weight
# Lưu ý: Bạn cần lưu kết quả dự đoán nhãn vào biến y_pred1
# YOUR CODE HERE
raise NotImplementedError()
lrmodel1 = ...
lrmodel1.fit(...)
y_pred1 = ...

In [ ]:
# Mô hình sử dụng class_weight='balanced'
# Lưu ý: Bạn cần lưu kết quả dự đoán nhãn vào biến y_pred2
# YOUR CODE HERE
raise NotImplementedError()
lrmodel2 = ...
lrmodel2.fit(...)
y_pred2 = ...

**Đánh giá hai mô hình**

Bạn cần lưu kết quả vào các biến sau:
- `acc1`: Accuracy của mô hình 1 (không dùng class_weight).
- `f_score1`: F1_score của mô hình 1 (không dùng class_weight).
- `acc2`: Accuracy của mô hình 2 (dùng class_weight).
- `f_score2`: F1_score của mô hình 2 (dùng class_weight).

In [ ]:
# YOUR CODE HERE
raise NotImplementedError()
acc1 = ...
f_score1 = ...
acc2 = ...
f_score2 = ...

In [ ]:
# Xem kết quả đánh giá
print("Không dùng class_weight")
print("Accuracy:", acc1)
print("F1-score:", f_score1)
print("\nDùng class_weight='balanced'")
print("Accuracy:", acc2)
print("F1-score:", f_score2)

In [ ]:
# TEST
# Cho phép sai khác nhỏ giữa các phiên bản thư viện và nghiệm tối ưu.
assert np.isclose(acc1, 0.8192, atol=0.002, rtol=0)
assert np.isclose(f_score1, 0.4556, atol=0.002, rtol=0)
assert np.isclose(acc2, 0.7737, atol=0.002, rtol=0)
assert np.isclose(f_score2, 0.5301, atol=0.002, rtol=0)


Đánh giá kết quả này. Theo bạn, nên sử dụng metrics nào trong trường hợp này?

_YOUR ANSWER HERE_

## Câu hỏi 2: Diễn giải mô hình Logistic Regression

Dữ liệu được cung cấp trong bài này có khá nhiều đặc trưng và hai mô hình bạn vừa cài đặt sử dụng tất cả các đặc trưng của dữ liệu này. Tuy nhiên, không phải đặc trưng nào cũng quan trọng, một số đặc trưng thậm chí có thể gây nhiễu.

Để khắc phục điều này, ta áp dụng Logistic Regression với L1 regularization. Kỹ thuật này không chỉ giúp mô hình tránh overfitting mà còn có khả năng loại bỏ tự động các đặc trưng không cần thiết bằng cách đẩy hệ số của chúng về 0. Nhờ đó, mô hình trở nên đơn giản, dễ diễn giải hơn và chỉ tập trung vào các đặc trưng thực sự quan trọng ảnh hưởng đến khả năng vỡ nợ của khách hàng.

Cài đặt: Điều chỉnh tham số của hàm [Logistic Regression](https://scikit-learn.org/stable/modules/generated/sklearn.linear_model.LogisticRegression.html)
- Đặt `penalty = 'l1'` để dùng L1 regularization (`default='l2'`).
- Đặt hàm tối ưu hóa (`solver = liblinear`): Scikit-learn chỉ hỗ trợ hàm tối ưu hóa liblinear và saga với penalty L1.
    - solver liblinear: Giải bài toán tối ưu hóa bằng cách tối ưu từng tham số (hệ số) một, trong khi giữ các tham số còn lại cố định. $\rightarrow$ Thích hợp cho các bài toán vừa và nhỏ.
    - solver saga: Là một phiên bản cải tiến của Stochastic Gradient Descent (SGD), tối ưu trên từng batch nhỏ. $\rightarrow$ Nhanh và hiệu quả cho dữ liệu lớn. Tuy nhiên, solver saga nhạy cảm với dữ liệu chưa chuẩn hóa và có thể dao động hoặc hội tụ chậm nếu siêu tham số không tốt.

Lưu ý: So sánh độ lớn hệ số cần xét thang đo. Biến liên tục đã được chuẩn hóa; hệ số biến one-hot biểu diễn thay đổi so với nhóm tham chiếu bị bỏ bởi `drop_first=True`. Vì vậy top 10 theo |hệ số| chỉ là cách xem hệ số của mô hình này, không phải thứ hạng ảnh hưởng nhân quả. Dấu hệ số được diễn giải khi giữ các biến khác cố định.


### Cài đặt mô hình
Lưu mô hình vào biến `lrmodel3`

In [ ]:
# YOUR CODE HERE
raise NotImplementedError()
lrmodel3 = ...
lrmodel3.fit(...)

### Liệt kê các đặc trưng có hệ số khác 0
Các đặc trưng có hệ số khác 0 là các đặc trưng không bị loại bỏ

In [ ]:
# Lấy hệ số
coef = lrmodel3.coef_[0]

# Lấy tên cột
non_zero_indices = np.where(coef != 0)[0]
non_zero_features = [(X_train.columns[i], coef[i]) for i in non_zero_indices]

print("Các đặc trưng có hệ số khác 0:")
for name, weight in non_zero_features:
    print(f"{name}: {weight:.4f}")

### Lấy top 10 đặc trưng theo độ lớn tuyệt đối của hệ số


In [ ]:
# Sắp xếp theo giá trị tuyệt đối
top_10 = sorted(non_zero_features, key=lambda x: abs(x[1]), reverse=True)[:10]
print("\nTop 10 đặc trưng theo |hệ số|:")
for name, weight in top_10:
    sign = "Tăng" if weight > 0 else "Giảm"
    print(f"{name:30s} | Coef: {weight:.4f} | {sign} xác suất vỡ nợ")

Giải thích ý nghĩa dấu của hệ số trong ngữ cảnh thực tế: Đặc trưng này có làm tăng hay giảm xác suất vỡ nợ?

_YOUR ANSWER HERE_

## Câu hỏi 3: Đưa ra chiến lược sàng lọc khách hàng rủi ro

### Chọn mô hình
Dùng mô hình có `class_weight='balanced'` (`lrmodel2`) từ Phần 1 để so sánh hai ngưỡng trên cùng mô hình. Thảo luận F1-score và recall khi so sánh với mô hình không dùng trọng số.

Đây là bài minh họa đánh đổi precision–recall trên tập test. Trong ứng dụng thực tế, chọn mô hình/ngưỡng trên tập validation rồi đánh giá cuối cùng trên tập test độc lập.


In [ ]:
# YOUR CODE HERE
raise NotImplementedError()
model = ...

In [ ]:
# Xác suất vỡ nợ trên tập test
y_proba = model.predict_proba(X_test)[:, 1]

Với ngưỡng 0.4, đánh dấu khách hàng là rủi ro cao.
Đếm số lượng bị đánh dấu.
Trong số đó, có bao nhiêu người thực sự đã vỡ nợ?
Tính precision và recall.

#### Ngưỡng 0.4
Cần lưu kết quả vào các biến sau:
- `num_flagged_04`: Số người bị đánh dấu là vỡ nợ
- `true_positives_04`: Số người bị đánh dấu và thực sự đã vỡ nợ (true positives)
- `precision_04`: precision, là số thực
- `recall_04`: recall, là số thực

In [ ]:
# YOUR CODE HERE
raise NotImplementedError()

In [ ]:
# Xem kết quả
print('Số người bị đánh dấu là vỡ nợ:', num_flagged_04)
print('Số người thực sự vỡ nợ:', true_positives_04)
print('Precision:', precision_04)
print('Recall:', recall_04)

In [ ]:
# TEST
# Đối chiếu với xác suất của chính mô hình, không cố định số lượng theo phiên bản thư viện.
expected_labels = (y_proba >= 0.4).astype(int)
assert num_flagged_04 == np.count_nonzero(expected_labels)
assert true_positives_04 == np.count_nonzero((expected_labels == 1) & (y_test == 1))
assert np.isclose(precision_04, precision_score(y_test, expected_labels, zero_division=0))
assert np.isclose(recall_04, recall_score(y_test, expected_labels, zero_division=0))


#### Ngưỡng 0.6
Cần lưu kết quả vào các biến sau:
- `num_flagged_06`: Số người bị đánh dấu là vỡ nợ
- `true_positives_06`: Số người bị đánh dấu và thực sự đã vỡ nợ (true positives)
- `precision_06`: precision, là số thực
- `recall_06`: recall, là số thực

In [ ]:
# YOUR CODE HERE
raise NotImplementedError()

In [ ]:
# Xem kết quả
print('Số người bị đánh dấu là vỡ nợ:', num_flagged_06)
print('Số người thực sự vỡ nợ:', true_positives_06)
print('Precision:', precision_06)
print('Recall:', recall_06)

In [ ]:
# TEST
# Đối chiếu với xác suất của chính mô hình, không cố định số lượng theo phiên bản thư viện.
expected_labels = (y_proba >= 0.6).astype(int)
assert num_flagged_06 == np.count_nonzero(expected_labels)
assert true_positives_06 == np.count_nonzero((expected_labels == 1) & (y_test == 1))
assert np.isclose(precision_06, precision_score(y_test, expected_labels, zero_division=0))
assert np.isclose(recall_06, recall_score(y_test, expected_labels, zero_division=0))


#### Kết luận
Ngân hàng nên chọn ngưỡng nào? Vì sao?

_YOUR ANSWER HERE_